In [1]:
import os
import math
import json
import random
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import numpy as np
import pandas as pd

import torch 
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

print('Torch:', torch.__version__)
print('Device:', device)
print(torch.device)

cpu
Torch: 2.2.2
Device: cpu
<class 'torch.device'>


In [2]:
DATA_PATH = 'Scats Data October 2006.xls'
SHEET = 'Data'

assert os.path.exists(DATA_PATH), f"Data file not found at {DATA_PATH}"

if DATA_PATH.lower().endswith('.xls') or DATA_PATH.lower().endswith('.xlsx'):
    df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET, header=1)
elif DATA_PATH.lower().endswith('.csv'):
    df_raw = pd.read_csv(DATA_PATH)
else:
    raise ValueError("Unsupported file format. Please provide an Excel (.xls/.xlsx) or CSV (.csv) file.")

print('Loaded:', DATA_PATH)
print('Rows:', len(df_raw), 'Cols:', df_raw.shape[1])
print('Columns:', list(df_raw.columns)[:15], '...')
df_raw.head()

Loaded: Scats Data October 2006.xls
Rows: 4192 Cols: 106
Columns: ['SCATS Number', 'Location', 'CD_MELWAY', 'NB_LATITUDE', 'NB_LONGITUDE', 'HF VicRoads Internal', 'VR Internal Stat', 'VR Internal Loc', 'NB_TYPE_SURVEY', 'Date', 'V00', 'V01', 'V02', 'V03', 'V04'] ...


,SCATS Number,Location,CD_MELWAY,NB_LATITUDE,NB_LONGITUDE,HF VicRoads Internal,VR Internal Stat,VR Internal Loc,NB_TYPE_SURVEY,Date,...,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95
0,970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-01 00:15:00,...,114,97,97,66,81,50,59,47,29,34
1,970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-02 00:15:00,...,111,102,107,114,80,60,62,48,44,26
2,970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-03 00:15:00,...,130,132,114,86,93,90,73,57,29,40
3,970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-04 00:15:00,...,115,113,132,101,113,90,78,66,52,44
4,970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-05 00:15:00,...,171,120,116,113,99,91,61,55,49,36


In [3]:
df = df_raw.copy()

if 'SCATS Number' in df.columns:
    df = df.rename(columns={'SCATS Number': 'SCATS_Number'})
    df['SCATS Number'] = pd.to_numeric(df['SCATS_Number'], errors='coerce').astype('Int64')
    
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    
flow_cols = [c for c in df.columns if isinstance(c, str) and c.startswith('V') and len(c) == 3]
flow_cols = sorted(flow_cols, key=lambda s: int(s[1:]))

print('Found flow columns:', len(flow_cols), flow_cols[:5], '...', flow_cols[-5:])

for c in flow_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
    
if 'SCATS Number' in df.columns:
    df = df[df['SCATS Number'].notna()].copy()
if 'Date' in df.columns:
    df = df[df['Date'].notna()].copy()
    
df = df.dropna(subset=flow_cols, how='all').copy()

df[flow_cols] = df[flow_cols].fillna(0)

df = df[df[flow_cols].sum(axis=1) > 0].copy()

for c in ['NB_LATITUDE', 'NB_LONGITUDE']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

print('After cleaning:', df.shape)
df[['SCATS Number', 'Date'] + flow_cols[:6]].head(3)

os.makedirs('artifacts_person1', exist_ok=True)
cols_for_clean_csv = [c for c in ['SCATS Number', 'Date', 'NB_LATITUDE', 'NB_LONGITUDE'] if c in df.columns] + flow_cols
df[cols_for_clean_csv].to_csv('artifacts_person1/cleaned_flows.csv', index=False)
print('Saved cleaned CSV -> artifacts_person1/cleaned_flows.csv')

Found flow columns: 96 ['V00', 'V01', 'V02', 'V03', 'V04'] ... ['V91', 'V92', 'V93', 'V94', 'V95']
After cleaning: (4192, 107)
Saved cleaned CSV -> artifacts_person1/cleaned_flows.csv


In [ ]:
seq_len = 4  # 12*15min = 3 hours history

# Split by SCATS site to reduce leakage
site_groups = df['SCATS Number'].astype(str)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(df, groups=site_groups))
df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)  # 0.25 of train -> val => val ~0.2 overall
tr_idx, val_idx = next(gss2.split(df_train, groups=df_train['SCATS Number'].astype(str)))
df_tr = df_train.iloc[tr_idx].copy()
df_val = df_train.iloc[val_idx].copy()

print('Split sizes:', len(df_tr), len(df_val), len(df_test))
print('Unique sites:', df['SCATS Number'].nunique(), 'train/val/test:', df_tr['SCATS Number'].nunique(), df_val['SCATS Number'].nunique(), df_test['SCATS Number'].nunique())

# Time-of-day features for 96 steps
T = 96
tod = np.arange(T)
tod_sin = np.sin(2 * np.pi * tod / T).astype(np.float32)
tod_cos = np.cos(2 * np.pi * tod / T).astype(np.float32)

# Scale flow values (fit on train only)
flow_scaler = StandardScaler()
flow_scaler.fit(df_tr[flow_cols].to_numpy().reshape(-1, 1))

def build_sequences_from_df(d: pd.DataFrame, seq_len: int):
    X_list = []
    y_list = []
    meta = []  # (site, date, step)

    flows = d[flow_cols].to_numpy(dtype=np.float32)  # shape (n_rows, 96)
    flows_scaled = flow_scaler.transform(flows.reshape(-1, 1)).reshape(flows.shape).astype(np.float32)

    sites = d['SCATS Number'].to_numpy()
    dates = d['Date'].to_numpy()

    for i in range(flows_scaled.shape[0]):
        series = flows_scaled[i]
        for t in range(0, T - seq_len):
            win = series[t:t + seq_len]
            # features per step: [flow, sin, cos]
            step_idx = np.arange(t, t + seq_len)
            X = np.stack([win, tod_sin[step_idx], tod_cos[step_idx]], axis=-1)
            y = series[t + seq_len]
            X_list.append(X)
            y_list.append(y)
            meta.append((int(sites[i]), pd.to_datetime(dates[i]), int(t + seq_len)))

    return np.asarray(X_list, dtype=np.float32), np.asarray(y_list, dtype=np.float32), meta

X_tr, y_tr, meta_tr = build_sequences_from_df(df_tr, seq_len)
X_val, y_val, meta_val = build_sequences_from_df(df_val, seq_len)
X_test, y_test, meta_test = build_sequences_from_df(df_test, seq_len)

print('X_tr:', X_tr.shape, 'y_tr:', y_tr.shape)
print('X_val:', X_val.shape, 'y_val:', y_val.shape)
print('X_test:', X_test.shape, 'y_test:', y_test.shape)

Split sizes: 2520 896 776
Unique sites: 40 train/val/test: 24 8 8
X_tr: (211680, 12, 3) y_tr: (211680,)
X_val: (75264, 12, 3) y_val: (75264,)
X_test: (65184, 12, 3) y_test: (65184,)


In [5]:
os.makedirs('artifacts_person1', exist_ok=True)

base_cols = [c for c in ['SCATS Number', 'Date', 'NB_LATITUDE', 'NB_LONGITUDE'] if c in df.columns]
cols_to_save = base_cols + flow_cols

df_tr[cols_to_save].to_csv('artifacts_person1/train_rows.csv', index=False)
df_val[cols_to_save].to_csv('artifacts_person1/val_rows.csv', index=False)
df_test[cols_to_save].to_csv('artifacts_person1/test_rows.csv', index=False)

print('Saved row-level splits:')
print('-', 'artifacts_person1/train_rows.csv', len(df_tr))
print('-', 'artifacts_person1/val_rows.csv', len(df_val))
print('-', 'artifacts_person1/test_rows.csv', len(df_test))

Saved row-level splits:
- artifacts_person1/train_rows.csv 2520
- artifacts_person1/val_rows.csv 896
- artifacts_person1/test_rows.csv 776


In [6]:
def inverse_flow_scale(x_scaled: np.ndarray) -> np.ndarray:
    x_scaled = np.asarray(x_scaled).reshape(-1, 1)
    return flow_scaler.inverse_transform(x_scaled).reshape(-1)

# For evaluation in original units
y_test_true = inverse_flow_scale(y_test)
print('Example y_test (scaled -> true):')
for i in range(5):
    print(float(y_test[i]), '->', float(y_test_true[i]))

Example y_test (scaled -> true):
-1.0235106945037842 -> 15.999995231628418
-1.0235106945037842 -> 15.999995231628418
-1.0578359365463257 -> 13.00000286102295
-1.0235106945037842 -> 15.999995231628418
-1.0235106945037842 -> 15.999995231628418


In [7]:
class LSTMFlowDataset(nn.Module):
    def __init__(self, n_features: int, hidden_size: int = 64, dense_size: int = 32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, batch_first=True)
        self.dense1 = nn.Linear(hidden_size, dense_size)
        self.relu = nn.ReLU()
        self.dense2 = nn.Linear(dense_size, 1)
        
    def forward(self, x):
        # x: (batch, seq_len, n_features)
        out, _ = self.lstm(x)
        out = out[:, -1, :]        # (batch, 64)
        out = self.dense1(out)     # (batch, 32)
        out = self.relu(out)
        out = self.dense2(out) 
        return out
    
lstm = LSTMFlowDataset(n_features=X_tr.shape[2])
print(lstm)

LSTMFlowDataset(
  (lstm): LSTM(3, 64, batch_first=True)
  (dense1): Linear(in_features=64, out_features=32, bias=True)
  (relu): ReLU()
  (dense2): Linear(in_features=32, out_features=1, bias=True)
)


In [ ]:
# Training loop (PyTorch)
batch_size = 64
epochs = 60
patience = 8  # early stopping
min_delta = 1e-6

train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last= True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)

criterion = nn.MSELoss()
lstm = lstm.to(device)
optimizer = torch.optim.Adam(lstm.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-6
)

best_val = float('inf')
best_state = None
wait = 0
history_lstm = []

for epoch in range(1, epochs + 1):
    lstm.train()
    train_losses = []
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        pred = lstm(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.detach().item())
        
    lstm.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = lstm(xb)
            loss = criterion(pred, yb)
            val_losses.append(loss.detach().item())

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    scheduler.step(val_loss)
    lr = float(optimizer.param_groups[0]['lr'])
    history_lstm.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'lr': lr})
    print(f"Epoch {epoch:03d} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | lr={lr:.2e}")
    
    if val_loss < (best_val - min_delta):
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in lstm.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch} (best val_loss={best_val:.5f})")
            break

if best_state is not None:
    lstm.load_state_dict(best_state)



/Users/mac/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 001 | train_loss=0.97441 | val_loss=0.82939 | lr=1.00e-03
Epoch 002 | train_loss=0.97389 | val_loss=0.82860 | lr=1.00e-03


In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(math.sqrt(mse))
    r2 = r2_score(y_true, y_pred)
    return {'MAE': float(mae), 'RMSE': float(rmse), 'R2': float(r2)}

lstm.eval()
with torch.no_grad():
    pred_lstm_scaled = lstm(torch.from_numpy(X_test).to(device)).detach().cpu().numpy().reshape(-1)

pred_lstm_true = inverse_flow_scale(pred_lstm_scaled)

metrics_lstm = regression_metrics(y_test_true, pred_lstm_true)
metrics_lstm

{'MAE': 67.8685302734375,
 'RMSE': 89.10334315680164,
 'R2': 0.022012054920196533}

In [ ]:
lstm.eval()
with torch.no_grad():
    pred_lstm_scaled = lstm(torch.from_numpy(X_test).to(device)).detach().cpu().numpy().reshape(-1)

pred_lstm_veh_15min = inverse_flow_scale(pred_lstm_scaled)
y_true_veh_15min = y_test_true

metrics_lstm = regression_metrics(y_true_veh_15min, pred_lstm_veh_15min)

results = pd.DataFrame([
    {'model': 'LSTM_torch', **metrics_lstm},
])

pred_df = pd.DataFrame({
    'scats_site': [m[0] for m in meta_test],
    'date': [m[1] for m in meta_test],
    'step_index_0to95': [m[2] for m in meta_test],
    'y_true_veh_15min': y_true_veh_15min,
    'pred_lstm_veh_15min': pred_lstm_veh_15min,
})

display(results)
pred_df.head()

,model,MAE,RMSE,R2
0,LSTM_torch,67.86853,89.103343,0.022012


,scats_site,date,step_index_0to95,y_true_veh_15min,pred_lstm_veh_15min
0,2827,2006-10-01 00:15:00,12,15.999995,115.552269
1,2827,2006-10-01 00:15:00,13,15.999995,115.552223
2,2827,2006-10-01 00:15:00,14,13.000003,115.559013
3,2827,2006-10-01 00:15:00,15,15.999995,115.543427
4,2827,2006-10-01 00:15:00,16,15.999995,115.560532


In [ ]:
os.makedirs('artifacts_person1', exist_ok=True)

# Save LSTM (PyTorch) model weights + minimal config
torch.save(lstm.state_dict(), 'artifacts_person1/lstm_flow_15min_torch.pt')
with open('artifacts_person1/lstm_flow_15min_torch_config.json', 'w', encoding='utf-8') as f:
    json.dump({
        'seq_len': int(seq_len),
        'n_features': int(X_tr.shape[-1]),
        'hidden_size': 64,
        'dense_size': 32,
        'backend': 'pytorch',
    }, f, ensure_ascii=False, indent=2)

# Save scaler (needed to convert model outputs back to vehicles/15min)
joblib.dump(flow_scaler, 'artifacts_person1/flow_scaler.joblib')

# Save results
pred_df.to_csv('artifacts_person1/predictions_test.csv', index=False)
results.to_csv('artifacts_person1/model_results.csv', index=False)

print('Saved artifacts to artifacts_person1/')

Saved artifacts to artifacts_person1/


In [ ]:
class GRURegressor(nn.Module):
    def __init__(self, n_features: int, hidden_size: int = 64, dense_size: int = 32):
        super().__init__()
        self.gru = nn.GRU(input_size=n_features, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, dense_size)
        self.fc2 = nn.Linear(dense_size, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        out, _ = self.gru(x)
        last = out[:, -1, :]
        x = self.relu(self.fc1(last))
        x = self.fc2(x)
        return x.squeeze(-1)

gru = GRURegressor(n_features=X_tr.shape[-1]).to(device)
print("GRU Model Summary:")
print(gru)

GRU Model Summary:
GRURegressor(
  (gru): GRU(3, 64, batch_first=True)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
)


In [ ]:
# Training GRU model (same hyperparameters as LSTM for fair comparison)
batch_size = 64
epochs = 60
patience = 8  # early stopping
min_delta = 1e-6

criterion_gru = nn.MSELoss()
optimizer_gru = torch.optim.Adam(gru.parameters(), lr=1e-3)
scheduler_gru = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_gru, mode='min', factor=0.5, patience=4, min_lr=1e-6
)

best_val_gru = float('inf')
best_state_gru = None
wait = 0
history_gru = []

print("Training GRU model...")
for epoch in range(1, epochs + 1):
    gru.train()
    train_losses = []
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer_gru.zero_grad(set_to_none=True)
        pred = gru(xb)
        loss = criterion_gru(pred, yb)
        loss.backward()
        optimizer_gru.step()
        train_losses.append(loss.detach().item())

    gru.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = gru(xb)
            loss = criterion_gru(pred, yb)
            val_losses.append(loss.detach().item())

    train_loss_gru = float(np.mean(train_losses))
    val_loss_gru = float(np.mean(val_losses))
    scheduler_gru.step(val_loss_gru)
    lr = float(optimizer_gru.param_groups[0]['lr'])
    history_gru.append({'epoch': epoch, 'train_loss': train_loss_gru, 'val_loss': val_loss_gru, 'lr': lr})
    
    if (epoch % 10) == 0 or epoch < 5:
        print(f"Epoch {epoch:03d} | train_loss={train_loss_gru:.5f} | val_loss={val_loss_gru:.5f} | lr={lr:.2e}")

    if val_loss_gru < (best_val_gru - min_delta):
        best_val_gru = val_loss_gru
        best_state_gru = {k: v.detach().cpu().clone() for k, v in gru.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch} (best val_loss={best_val_gru:.5f})")
            break

if best_state_gru is not None:
    gru.load_state_dict(best_state_gru)

# Evaluate GRU on test set
gru.eval()
with torch.no_grad():
    pred_gru_scaled = gru(torch.from_numpy(X_test).to(device)).detach().cpu().numpy().reshape(-1)

pred_gru_true = inverse_flow_scale(pred_gru_scaled)
metrics_gru = regression_metrics(y_test_true, pred_gru_true)
print("\nGRU Test Metrics:", metrics_gru)

Training GRU model...
Epoch 001 | train_loss=0.07184 | val_loss=0.04354 | lr=1.00e-03
Epoch 002 | train_loss=0.06025 | val_loss=0.04109 | lr=1.00e-03
Epoch 003 | train_loss=0.05869 | val_loss=0.04188 | lr=1.00e-03
Epoch 004 | train_loss=0.05772 | val_loss=0.04164 | lr=1.00e-03
Epoch 010 | train_loss=0.05483 | val_loss=0.04156 | lr=1.00e-03
Epoch 020 | train_loss=0.05086 | val_loss=0.04016 | lr=5.00e-04
Early stopping at epoch 27 (best val_loss=0.03933)

GRU Test Metrics: {'MAE': 13.573715209960938, 'RMSE': 20.536499281386178, 'R2': 0.94804847240448}
